# Phase 1 — Backpropagation & MLP (순수 NumPy)

자동미분 없이 체인 룰(Chain Rule)을 손으로 작성한 MLP로 Fashion-MNIST를 학습한다.

목표:
1. 역전파의 각 항(`Dense`, `Sigmoid`, `SoftmaxCrossEntropy`)의 미분을 코드로 확인
2. **수치 미분과 역전파 결과 비교**로 구현 검증 (rel err < 1e-6)
3. 깊이 3/5/10층 Sigmoid 네트워크에서 **Vanishing Gradient** 관찰
4. ReLU(He init)가 이를 어떻게 완화하는지 확인

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np

from dl_history.data import INPUT_DIM, NUM_CLASSES, load_fashion_mnist_numpy
from dl_history.metrics import EpochHistory, accuracy
from dl_history.models.mlp_numpy import MLP, check_gradients
from dl_history.viz import plot_accuracy_curves, plot_gradient_norms, plot_loss_curves

rng = np.random.default_rng(0)

## 1. 데이터: Fashion-MNIST

28×28 흑백 이미지를 784차원 벡터로 flatten한 뒤,
train 통계(mean/std)로 **표준화**한다 — zero-centered 입력은 깊은 ReLU 네트워크의
학습 안정성에 필수적이다 (raw [0,1] 입력에서는 깊은 ReLU가 발산한다).

In [2]:
x_train, y_train, x_test, y_test = load_fashion_mnist_numpy()

# 표준화: zero-centered 입력이 깊은 ReLU 네트워크의 안정성에 필수적
# (raw [0,1] 입력에서는 실용적인 learning rate에서 발산한다)
mean, std = x_train.mean(), x_train.std()
x_train = (x_train - mean) / std
x_test = (x_test - mean) / std

print(x_train.shape, x_test.shape, round(x_train.mean(), 4), round(x_train.std(), 4))
print("classes:", np.bincount(y_train))

(60000, 784) (10000, 784) -0.0 1.0
classes: [6000 6000 6000 6000 6000 6000 6000 6000 6000 6000]


## 2. 역전파 검증 — 수치 미분 vs 체인 룰

역전파 구현이 맞는지 **중앙차분 수치 미분**과 비교한다.
체인 룰이 정확하면 상대오차는 1e-6 이하로 떨어진다.

In [3]:
for activation in ("sigmoid", "tanh", "relu"):
    model = MLP.build(8, [16, 12], 5, activation=activation, seed=0)
    batch_x = rng.standard_normal((16, 8))
    batch_y = rng.integers(0, 5, size=16)
    rel_err = check_gradients(model, batch_x, batch_y, eps=1e-5, num_checks=10)
    print(f"{activation:>8}: max relative error = {rel_err:.2e}")

 sigmoid: max relative error = 3.08e-08
    tanh: max relative error = 7.85e-09
    relu: max relative error = 3.18e-09


## 3. 실험 A — 깊이와 Vanishing Gradient

Sigmoid의 미분은 |σ′| ≤ 0.25. 역전파는 층을 지날 때마다 이 계수(과 W.T의 gain)가 곱해지므로,
10층이면 초기층 gradient가 지수적으로 사라진다. 층별 ‖dW‖₂를 기록해 확인한다.

In [4]:
BATCH = 128
EPOCHS = 10
LR = 0.1
WIDTH = 128

def train(model, epochs=EPOCHS, lr=LR):
    history = EpochHistory()
    for _ in range(epochs):
        epoch_loss = 0.0
        n_batches = 0
        for start in range(0, x_train.shape[0], BATCH):
            end = min(start + BATCH, x_train.shape[0])
            epoch_loss += model.train_batch(x_train[start:end], y_train[start:end], lr)
            n_batches += 1
        # predict()는 이미 argmax 결과를 반환하므로 정확도는 직접 비교
        val_acc = float(np.mean(model.predict(x_test) == y_test))
        train_acc = float(np.mean(model.predict(x_train[:10000]) == y_train[:10000]))
        history.record(epoch_loss / n_batches, train_acc, float("nan"), val_acc)
    return history

hist_a, grads_a, labels_a = {}, {}, {}
for name, hidden in [
    ("3-layer", [WIDTH, WIDTH]),
    ("5-layer", [WIDTH] * 4),
    ("10-layer", [WIDTH] * 9),
]:
    model = MLP.build(INPUT_DIM, hidden, NUM_CLASSES, activation="sigmoid", seed=0)
    print(f"training {name} sigmoid ...")
    hist_a[name] = train(model)
    grads_a[name] = model.gradient_history
    labels_a[name] = model.layer_names()
    print(f"  final val acc = {hist_a[name].val_acc[-1]:.4f}")

training 3-layer sigmoid ...
  final val acc = 0.8578
training 5-layer sigmoid ...
  final val acc = 0.8157
training 10-layer sigmoid ...
  final val acc = 0.1000


In [5]:
plot_gradient_norms(grads_a, labels_a, "Exp A — 층별 gradient norm (Sigmoid, log scale)", "phase1_expA_gradnorm.png")
plot_loss_curves(hist_a, "Exp A — 깊이별 loss", "phase1_expA_loss.png")

# 10층 네트워크에서 첫 층 vs 마지막 층 gradient norm 비교
means = grads_a["10-layer"].mean_per_layer()
print(f"layer 1: {means[0]:.3e}   layer 10: {means[-1]:.3e}   ratio: {means[0] / means[-1]:.1e}")

Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


layer 1: 9.934e-06   layer 10: 4.486e-01   ratio: 2.2e-05


## 4. 실험 B — Sigmoid vs ReLU

동일한 10층 구조에서 ReLU(He init)로 바꾸면 미분이 1 또는 0이므로
gradient가 지수적으로 사라지지 않는다. 수렴 속도를 비교한다.

In [6]:
hist_b = {}
for label, activation in [("sigmoid", "sigmoid"), ("relu", "relu")]:
    model = MLP.build(INPUT_DIM, [WIDTH] * 9, NUM_CLASSES, activation=activation, seed=0)
    print(f"training 10-layer {label} ...")
    hist_b[label] = train(model)
    print(f"  final val acc = {hist_b[label].val_acc[-1]:.4f}")

plot_loss_curves(hist_b, "Exp B — 10층 Sigmoid vs ReLU (loss)", "phase1_expB_loss.png")
plot_accuracy_curves(hist_b, "Exp B — 10층 Sigmoid vs ReLU (val acc)", "phase1_expB_acc.png")

training 10-layer sigmoid ...
  final val acc = 0.1000
training 10-layer relu ...


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], 

  final val acc = 0.8758


PosixPath('/Users/nixos/workspace/ai-history-with-code/results/phase1_expB_acc.png')

## 5. 실험 C — 2층 baseline

깊이 없이 2층(hidden 128)만으로도 ~88% 수준의 검증 정확도가 나온다.
깊이가 아니라 구조(Phase 2 이후의 CNN)가 이미지 과제의 진전을 만든다는 출발점.

In [7]:
model = MLP.build(INPUT_DIM, [WIDTH], NUM_CLASSES, activation="relu", seed=0)
hist_c = train(model)
print(f"2-layer baseline val acc = {hist_c.val_acc[-1]:.4f}")
plot_accuracy_curves({"2-layer baseline": hist_c}, "Exp C — 2층 baseline", "phase1_expC_acc.png")

2-layer baseline val acc = 0.8769


PosixPath('/Users/nixos/workspace/ai-history-with-code/results/phase1_expC_acc.png')